# A/B Testing & Statistical Significance Analysis

**Dataset:** Marketing A/B Testing (Kaggle, Faviovaz) — 588,101 users, comparing ad exposure vs. a PSA control.

**Pipeline:**
1. Data loading & SRM (Sample Ratio Mismatch) check
2. Hypothesis testing (Z-test for proportions)
3. Bootstrap validation
4. Business impact translation
5. Power analysis / MDE
6. CUPED variance reduction (simulated pre-period covariate)
7. Covariate adjustment + HTE via logistic regression interaction terms


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from scipy.optimize import brentq
import statsmodels.formula.api as smf

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv('../data/marketing_AB.csv')
df.columns = [c.replace(' ', '_') for c in df.columns]
df.head()

,Unnamed:_0,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


## 1. Sample Ratio Mismatch (SRM) Check

In [2]:
observed = df['test_group'].value_counts()[['ad','psa']].values
total = observed.sum()
expected_ratio = [0.96, 0.04]
expected = [total * r for r in expected_ratio]

chi2, p = chisquare(f_obs=observed, f_exp=expected)
print(f'Observed: {observed}')
print(f'Expected: {[round(e) for e in expected]}')
print(f'Chi-square: {chi2:.4f}, p-value: {p:.6f}')
print(f'SRM detected (p<0.001): {p < 0.001}')

Observed: [564577  23524]
Expected: [564577, 23524]
Chi-square: 0.0000, p-value: 0.999788
SRM detected (p<0.001): False


## 2. Hypothesis Test (Z-test for Proportions)

In [3]:
ad = df[df['test_group']=='ad']['converted']
psa = df[df['test_group']=='psa']['converted']

count = np.array([ad.sum(), psa.sum()])
nobs = np.array([len(ad), len(psa)])
z_stat, p_val = proportions_ztest(count, nobs)

p_ad, p_psa = ad.mean(), psa.mean()
abs_lift = p_ad - p_psa
rel_lift = abs_lift / p_psa * 100

se = np.sqrt(p_ad*(1-p_ad)/nobs[0] + p_psa*(1-p_psa)/nobs[1])
ci_low, ci_high = abs_lift - 1.96*se, abs_lift + 1.96*se

print(f'ad conversion:  {p_ad:.4%}')
print(f'psa conversion: {p_psa:.4%}')
print(f'Absolute lift: {abs_lift:.4%} | Relative lift: {rel_lift:.2f}%')
print(f'Z-stat: {z_stat:.4f}, p-value: {p_val:.10f}')
print(f'95% CI: [{ci_low:.4%}, {ci_high:.4%}]')

ad conversion:  2.5547%
psa conversion: 1.7854%
Absolute lift: 0.7692% | Relative lift: 43.09%
Z-stat: 7.3701, p-value: 0.0000000000
95% CI: [0.5951%, 0.9434%]


## 3. Bootstrap Validation (10,000 resamples)

In [4]:
ad_arr = ad.values.astype(int)
psa_arr = psa.values.astype(int)

n_boot = 10000
boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    ad_sample = np.random.choice(ad_arr, size=len(ad_arr), replace=True)
    psa_sample = np.random.choice(psa_arr, size=len(psa_arr), replace=True)
    boot_diffs[i] = ad_sample.mean() - psa_sample.mean()

ci_low_b, ci_high_b = np.percentile(boot_diffs, [2.5, 97.5])
print(f'Bootstrap mean lift: {boot_diffs.mean():.4%}')
print(f'Bootstrap 95% CI: [{ci_low_b:.4%}, {ci_high_b:.4%}]')
print(f'% samples with positive lift: {(boot_diffs > 0).mean():.4%}')

Bootstrap mean lift: 0.7702%
Bootstrap 95% CI: [0.5927%, 0.9409%]
% samples with positive lift: 100.0000%


## 4. Business Impact

In [5]:
ad_df = df[df['test_group']=='ad']
n_ad = len(ad_df)
actual_conversions = ad_df['converted'].sum()
expected_at_psa_rate = n_ad * p_psa
extra_conversions = actual_conversions - expected_at_psa_rate

print(f'Ad group size: {n_ad:,}')
print(f'Actual conversions: {int(actual_conversions):,}')
print(f'Expected at psa rate: {expected_at_psa_rate:,.0f}')
print(f'Extra conversions attributable to ads: {extra_conversions:,.0f}')

Ad group size: 564,577
Actual conversions: 14,423
Expected at psa rate: 10,080
Extra conversions attributable to ads: 4,343


## 5. Power Analysis / MDE

In [6]:
analysis = NormalIndPower()
ratio = len(ad) / len(psa)

mde_effect_size = analysis.solve_power(effect_size=None, nobs1=len(psa), alpha=0.05, power=0.80, ratio=ratio)

def h_diff(p_ad_guess):
    return proportion_effectsize(p_ad_guess, p_psa) - mde_effect_size

mde_p_ad = brentq(h_diff, p_psa, 0.99)
mde_abs = mde_p_ad - p_psa

print(f'Baseline (psa) conversion: {p_psa:.4%}')
print(f'MDE (80% power, alpha=0.05): {mde_abs:.4%}')
print(f'Observed lift: {p_ad - p_psa:.4%} (well above MDE -> well-powered)')

Baseline (psa) conversion: 1.7854%
MDE (80% power, alpha=0.05): 0.2552%
Observed lift: 0.7692% (well above MDE -> well-powered)


## 6. CUPED Variance Reduction

**Note:** this dataset has no genuine pre-experiment covariate. A pre-period engagement score is simulated (correlated with `converted`) purely to demonstrate the CUPED mechanism. In production, use a real pre-experiment metric (e.g., prior purchase history).

In [7]:
noise = np.random.normal(0, 1, len(df))
df['pre_sim'] = 4.0 * df['converted'].astype(int) + noise

ad_c = df[df['test_group']=='ad']
psa_c = df[df['test_group']=='psa']

Y_all = df['converted'].astype(int).values
X_all = df['pre_sim'].values
theta = np.cov(Y_all, X_all)[0,1] / np.var(X_all)

def adjust(sub):
    Y = sub['converted'].astype(int).values
    X = sub['pre_sim'].values
    return Y - theta*(X - X_all.mean())

ad_cuped = adjust(ad_c)
psa_cuped = adjust(psa_c)

se_orig = np.sqrt(p_ad*(1-p_ad)/len(ad_c) + p_psa*(1-p_psa)/len(psa_c))
ci_width_orig = 2*1.96*se_orig

se_cuped = np.sqrt(ad_cuped.var()/len(ad_cuped) + psa_cuped.var()/len(psa_cuped))
ci_width_cuped = 2*1.96*se_cuped

reduction_pct = (1 - ad_cuped.var()/ad_c['converted'].astype(int).var())*100
ci_narrowed_pct = (1 - ci_width_cuped/ci_width_orig)*100

print(f'Theta: {theta:.4f}')
print(f'Variance reduction: {reduction_pct:.1f}%')
print(f'CI width before: {ci_width_orig:.4%} | after: {ci_width_cuped:.4%}')
print(f'CI narrowed by: {ci_narrowed_pct:.1f}%')

Theta: 0.0706
Variance reduction: 28.5%
CI width before: 0.3483% | after: 0.3107%
CI narrowed by: 10.8%


## 7. Covariate Adjustment + HTE (Interaction Terms)

In [8]:
df_reg = df.copy()
df_reg['treatment'] = (df_reg['test_group']=='ad').astype(int)
df_reg['converted'] = df_reg['converted'].astype(int)

model1 = smf.logit('converted ~ treatment + total_ads', data=df_reg).fit(disp=0)
print('=== Covariate Adjustment (additive) ===')
print(model1.summary2().tables[1][['Coef.','P>|z|']])

=== Covariate Adjustment (additive) ===


              Coef.         P>|z|
Intercept -4.430664  0.000000e+00
treatment  0.389428  3.970893e-14
total_ads  0.010145  0.000000e+00


In [9]:
model2 = smf.logit('converted ~ treatment * total_ads', data=df_reg).fit(disp=0)
print('=== Interaction Term (HTE by ad exposure) ===')
print(model2.summary2().tables[1][['Coef.','P>|z|']])

=== Interaction Term (HTE by ad exposure) ===


                        Coef.         P>|z|
Intercept           -4.403465  0.000000e+00
treatment            0.361307  1.604385e-09
total_ads            0.009695  8.353438e-81
treatment:total_ads  0.000467  3.682584e-01


## Summary

In [10]:
summary = (
    f"SRM check: no mismatch detected (p={p:.4f})\n"
    f"Conversion lift: {rel_lift:.1f}% relative ({abs_lift:.2%} absolute), p<0.0001\n"
    f"95% CI: [{ci_low:.2%}, {ci_high:.2%}], confirmed via bootstrap\n"
    f"Extra conversions attributable to ads: {extra_conversions:,.0f}\n"
    f"Well-powered: observed lift ({p_ad-p_psa:.2%}) >> MDE ({mde_abs:.2%})\n"
    f"CUPED (simulated strong pre-period signal): {reduction_pct:.1f}% variance reduction\n"
    f'HTE by ad exposure: not significant (p={model2.pvalues["treatment:total_ads"]:.2f}) - effect stable across exposure levels'
)
print(summary)

SRM check: no mismatch detected (p=0.9998)
Conversion lift: 43.1% relative (0.77% absolute), p<0.0001
95% CI: [0.60%, 0.94%], confirmed via bootstrap
Extra conversions attributable to ads: 4,343
Well-powered: observed lift (0.77%) >> MDE (0.26%)
CUPED (simulated strong pre-period signal): 28.5% variance reduction
HTE by ad exposure: not significant (p=0.37) - effect stable across exposure levels
